# MCS603 — Programming II (Python)
## Week 14: Capstone Project — Integration

---
### Overview
The capstone brings together every topic from the course:

| Week | Topic | Used in Capstone |
|---|---|---|
| 1–2 | Syntax, variables, operators | Throughout |
| 3–4 | Control flow & loops | Logic, menus, data processing |
| 5–6 | Data structures | In-memory caches, comprehensions |
| 7 | Modules & packages | Modular architecture |
| 8 | File handling | CSV import/export, logging |
| 9 | Exception handling | Robustness throughout |
| 10–11 | Flask web framework | Web interface |
| 12–13 | Database (SQLite) | Persistent storage |

---
### Capstone Project: Student Portal Web Application

Build a **fully integrated Student Portal** using Flask + SQLite with:
- A modular Python package structure
- A relational SQLite database
- A Flask web front-end with Jinja2 templates
- CSV import/export
- Comprehensive exception handling and logging

### Outline
1. Project Architecture  
2. Database Layer  
3. Business Logic Layer  
4. Web Layer (Flask)  
5. File I/O & Logging  
6. Putting It All Together  
7. Group Project Brief  
8. Presentation Guidelines  

---
## 1. Project Architecture

```
student_portal/
│
├── app.py                   ← Flask application & routes
├── config.py                ← Configuration constants
│
├── db/
│   ├── __init__.py
│   ├── connection.py        ← Database connection helper
│   └── schema.sql           ← Table definitions
│
├── models/
│   ├── __init__.py
│   ├── student.py           ← Student CRUD
│   ├── course.py            ← Course CRUD
│   └── enrollment.py        ← Enrollment CRUD
│
├── utils/
│   ├── __init__.py
│   ├── validators.py        ← Input validation
│   ├── csv_io.py            ← Import/export CSV
│   └── logger.py            ← Application logging
│
├── templates/
│   ├── base.html
│   ├── home.html
│   ├── students/
│   │   ├── list.html
│   │   ├── detail.html
│   │   └── form.html
│   └── courses/
│       ├── list.html
│       └── detail.html
│
├── static/
│   └── style.css
│
├── data/
│   └── students_import.csv
│
└── logs/
    └── portal.log
```

---
## 2. Database Layer

In [ ]:
import os

# Create directory structure
for d in ["portal/db", "portal/models", "portal/utils",
          "portal/templates/students", "portal/templates/courses",
          "portal/static", "portal/data", "portal/logs"]:
    os.makedirs(d, exist_ok=True)
    init = os.path.join(d.split("/")[0], d.split("/")[1] if "/" in d else "", "__init__.py")

for pkg in ["portal/db", "portal/models", "portal/utils"]:
    with open(f"{pkg}/__init__.py", "w") as f:
        f.write("")

print("Directory structure created.")

In [ ]:
schema = """
CREATE TABLE IF NOT EXISTS students (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    name       TEXT NOT NULL,
    email      TEXT UNIQUE NOT NULL,
    dob        DATE,
    phone      TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS courses (
    code       TEXT PRIMARY KEY,
    title      TEXT NOT NULL,
    credits    INTEGER NOT NULL DEFAULT 4,
    capacity   INTEGER NOT NULL DEFAULT 30,
    lecturer   TEXT
);

CREATE TABLE IF NOT EXISTS enrollments (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    student_id  INTEGER NOT NULL REFERENCES students(id) ON DELETE CASCADE,
    course_code TEXT    NOT NULL REFERENCES courses(code),
    grade       INTEGER CHECK(grade IS NULL OR (grade BETWEEN 0 AND 100)),
    enrolled_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(student_id, course_code)
);
"""

with open("portal/db/schema.sql", "w") as f:
    f.write(schema)

print("schema.sql written.")

In [ ]:
connection_code = '''
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent.parent / "data" / "portal.db"

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def init_db():
    schema = (Path(__file__).parent / "schema.sql").read_text()
    with get_connection() as conn:
        conn.executescript(schema)
'''

with open("portal/db/connection.py", "w") as f:
    f.write(connection_code)

print("db/connection.py written.")

---
## 3. Models Layer

In [ ]:
student_model = '''
from db.connection import get_connection

def get_all():
    with get_connection() as conn:
        return [dict(r) for r in conn.execute("SELECT * FROM students ORDER BY name").fetchall()]

def get_by_id(sid):
    with get_connection() as conn:
        row = conn.execute("SELECT * FROM students WHERE id=?", (sid,)).fetchone()
        return dict(row) if row else None

def search(query):
    with get_connection() as conn:
        rows = conn.execute(
            "SELECT * FROM students WHERE name LIKE ? OR email LIKE ?",
            (f"%{query}%", f"%{query}%")
        ).fetchall()
        return [dict(r) for r in rows]

def create(name, email, dob=None, phone=None):
    with get_connection() as conn:
        cursor = conn.execute(
            "INSERT INTO students (name, email, dob, phone) VALUES (?,?,?,?)",
            (name, email, dob, phone)
        )
        return cursor.lastrowid

def update(sid, **fields):
    allowed = {"name", "email", "dob", "phone"}
    fields  = {k: v for k, v in fields.items() if k in allowed}
    if not fields:
        return 0
    cols = ", ".join(f"{k}=?" for k in fields)
    with get_connection() as conn:
        cursor = conn.execute(f"UPDATE students SET {cols} WHERE id=?",
                              (*fields.values(), sid))
        return cursor.rowcount

def delete(sid):
    with get_connection() as conn:
        cursor = conn.execute("DELETE FROM students WHERE id=?", (sid,))
        return cursor.rowcount
'''

with open("portal/models/student.py", "w") as f:
    f.write(student_model)

print("models/student.py written.")

In [ ]:
# utils/logger.py
logger_code = '''
from datetime import datetime
from pathlib import Path

LOG_PATH = Path(__file__).parent.parent / "logs" / "portal.log"

def log(level, message):
    ts  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    LOG_PATH.parent.mkdir(exist_ok=True)
    with open(LOG_PATH, "a") as f:
        f.write(f"[{ts}] [{level:<7}] {message}\\n")

def info(msg):    log("INFO",    msg)
def warning(msg): log("WARNING", msg)
def error(msg):   log("ERROR",   msg)
'''

# utils/validators.py
validators_code = '''
def validate_email(email):
    return "@" in email and "." in email.split("@")[-1]

def validate_grade(grade):
    try:
        g = int(grade)
        return 0 <= g <= 100
    except (TypeError, ValueError):
        return False

def validate_student(name, email):
    errors = []
    if not name or len(name.strip()) < 2:
        errors.append("Name must be at least 2 characters")
    if not validate_email(email):
        errors.append("Invalid email address")
    return errors
'''

# utils/csv_io.py
csv_io_code = '''
import csv
from pathlib import Path
from db.connection import get_connection
from utils.logger import info, error

def export_students(filepath):
    with get_connection() as conn:
        rows = conn.execute("SELECT id, name, email, dob, phone FROM students").fetchall()
    with open(filepath, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["id","name","email","dob","phone"])
        writer.writeheader()
        writer.writerows([dict(r) for r in rows])
    info(f"Exported {len(rows)} students to {filepath}")
    return len(rows)

def import_students(filepath):
    imported, failed = 0, 0
    with get_connection() as conn:
        with open(filepath, newline="") as f:
            for row in csv.DictReader(f):
                try:
                    conn.execute(
                        "INSERT INTO students (name, email, dob, phone) VALUES (?,?,?,?)",
                        (row["name"], row["email"], row.get("dob"), row.get("phone"))
                    )
                    imported += 1
                except Exception as e:
                    error(f"Import failed for row {row}: {e}")
                    failed += 1
    info(f"Import: {imported} succeeded, {failed} failed")
    return imported, failed
'''

for path, code in [
    ("portal/utils/logger.py",     logger_code),
    ("portal/utils/validators.py", validators_code),
    ("portal/utils/csv_io.py",     csv_io_code),
]:
    with open(path, "w") as f:
        f.write(code)

print("Utility modules written.")

---
## 4. Web Layer (Flask Routes)

In [ ]:
app_py = '''
from flask import Flask, render_template, request, redirect, url_for, flash
from db.connection import init_db
from models import student as StudentModel
from utils import validators, csv_io
from utils.logger import info, error
import sqlite3

app = Flask(__name__)
app.secret_key = "mcs603-secret"

@app.before_first_request
def setup():
    init_db()
    info("Application started")

# ── Home ─────────────────────────────────────────────────────────────
@app.route("/")
def home():
    students = StudentModel.get_all()
    return render_template("home.html", count=len(students))

# ── Students ─────────────────────────────────────────────────────────
@app.route("/students")
def student_list():
    q        = request.args.get("q", "")
    students = StudentModel.search(q) if q else StudentModel.get_all()
    return render_template("students/list.html", students=students, query=q)

@app.route("/students/<int:sid>")
def student_detail(sid):
    student = StudentModel.get_by_id(sid)
    if not student:
        flash("Student not found", "error")
        return redirect(url_for("student_list"))
    return render_template("students/detail.html", student=student)

@app.route("/students/add", methods=["GET", "POST"])
def student_add():
    if request.method == "POST":
        name  = request.form.get("name", "").strip()
        email = request.form.get("email", "").strip()
        dob   = request.form.get("dob") or None
        phone = request.form.get("phone") or None
        errors = validators.validate_student(name, email)
        if errors:
            for e in errors: flash(e, "error")
            return render_template("students/form.html", action="add")
        try:
            sid = StudentModel.create(name, email, dob, phone)
            info(f"Student added: {name} (id={sid})")
            flash(f"Student {name} added.", "success")
            return redirect(url_for("student_detail", sid=sid))
        except sqlite3.IntegrityError:
            flash("Email already registered.", "error")
    return render_template("students/form.html", action="add")

@app.route("/students/<int:sid>/delete", methods=["POST"])
def student_delete(sid):
    student = StudentModel.get_by_id(sid)
    if student:
        StudentModel.delete(sid)
        info(f"Student deleted: id={sid}")
        flash(f"Student removed.", "success")
    return redirect(url_for("student_list"))

# ── CSV Export ────────────────────────────────────────────────────────
@app.route("/export")
def export():
    path  = "data/export.csv"
    count = csv_io.export_students(path)
    flash(f"Exported {count} students to {path}", "success")
    return redirect(url_for("student_list"))

if __name__ == "__main__":
    app.run(debug=True)
'''

with open("portal/app.py", "w") as f:
    f.write(app_py)

print("portal/app.py written.")

---
## 5. Templates

In [ ]:
base_html = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>{% block title %}Student Portal{% endblock %}</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>
<header>
    <h1>MCS603 Student Portal</h1>
    <nav>
        <a href="/">Home</a>
        <a href="/students">Students</a>
        <a href="/students/add">Add Student</a>
        <a href="/export">Export CSV</a>
    </nav>
</header>
<main>
{% with messages = get_flashed_messages(with_categories=true) %}
{% if messages %}
  {% for cat, msg in messages %}
    <div class="alert {{ cat }}">{{ msg }}</div>
  {% endfor %}
{% endif %}
{% endwith %}
{% block content %}{% endblock %}
</main>
</body>
</html>
'''

list_html = '''{% extends "base.html" %}
{% block title %}Students{% endblock %}
{% block content %}
<h2>Students
  <small>({{ students | length }} found)</small>
</h2>
<form method="GET">
  <input name="q" value="{{ query }}" placeholder="Search name or email">
  <button type="submit">Search</button>
</form>
<table>
<tr><th>ID</th><th>Name</th><th>Email</th><th></th></tr>
{% for s in students %}
<tr>
  <td>{{ s.id }}</td>
  <td><a href="/students/{{ s.id }}">{{ s.name }}</a></td>
  <td>{{ s.email }}</td>
  <td>
    <form method="POST" action="/students/{{ s.id }}/delete"
          onsubmit="return confirm('Delete {{ s.name }}?')">
      <button type="submit">Delete</button>
    </form>
  </td>
</tr>
{% else %}
<tr><td colspan="4">No students found.</td></tr>
{% endfor %}
</table>
{% endblock %}
'''

form_html = '''{% extends "base.html" %}
{% block title %}{{ "Add" if action == "add" else "Edit" }} Student{% endblock %}
{% block content %}
<h2>{{ "Add" if action == "add" else "Edit" }} Student</h2>
<form method="POST">
  <label>Name: <input type="text" name="name" required></label>
  <label>Email: <input type="email" name="email" required></label>
  <label>Date of Birth: <input type="date" name="dob"></label>
  <label>Phone: <input type="tel" name="phone"></label>
  <button type="submit">Save</button>
</form>
{% endblock %}
'''

style_css = '''body { font-family: Arial, sans-serif; max-width: 960px;
        margin: 0 auto; padding: 20px; color: #333; }
header { background: #003366; color: white; padding: 15px 20px;
         border-radius: 6px; margin-bottom: 20px; }
header h1 { margin: 0 0 8px 0; font-size: 1.4em; }
nav a { color: #cce; margin-right: 15px; text-decoration: none; }
table { border-collapse: collapse; width: 100%; }
th, td { border: 1px solid #ddd; padding: 8px 12px; }
th { background: #f5f5f5; }
.alert { padding: 10px; border-radius: 4px; margin: 10px 0; }
.alert.success { background: #d4edda; color: #155724; }
.alert.error { background: #f8d7da; color: #721c24; }
label { display: block; margin: 8px 0; }
input { padding: 6px; width: 100%; max-width: 400px; }
button { background: #003366; color: white; padding: 8px 20px;
         border: none; border-radius: 4px; cursor: pointer; }
'''

for path, content in [
    ("portal/templates/base.html",          base_html),
    ("portal/templates/students/list.html",  list_html),
    ("portal/templates/students/form.html",  form_html),
    ("portal/static/style.css",              style_css),
]:
    with open(path, "w") as f:
        f.write(content)

print("Templates and CSS written.")
print("\nTo run the portal:")
print("  cd portal")
print("  pip install flask")
print("  python app.py")
print("  → http://127.0.0.1:5000")

---
## 6. Group Project Brief

### Project Options (choose one per group of 3–4 students)

**Option A — Library Management System**  
Books, members, borrowing records. Features: search, borrow/return, overdue report, fine calculation.

**Option B — Hostel Management System**  
Rooms, residents, payments. Features: room allocation, payment tracking, occupancy report.

**Option C — Exam Results System**  
Students, subjects, results. Features: result entry, transcript generation, pass/fail analysis, grade distribution chart.

**Option D — Clinic Appointment System**  
Patients, doctors, appointments. Features: book/cancel appointments, patient history, daily schedule.

---
### Minimum Requirements (all options)

| Requirement | Details |
|---|---|
| Module structure | At least 4 Python modules in a package |
| Database | SQLite with at least 3 related tables |
| CRUD | Full create/read/update/delete for main entity |
| File I/O | CSV import OR export of at least one entity |
| Exception handling | All DB and file ops protected |
| Logging | All user actions logged to file |
| Web interface | Flask with at least 5 routes and templates |
| Reports | At least 2 summary/analytics views |

---
### Grading Rubric (100 pts)

| Component | Points |
|---|---|
| Functionality — all features work correctly | 30 |
| Code quality — clean, modular, documented | 20 |
| Database design — correct schema, constraints | 15 |
| Exception handling & robustness | 10 |
| Web UI — usable, templates correct | 10 |
| Presentation — clear demo, can answer questions | 10 |
| Report document | 5 |
| **Total** | **100** |

---
## 7. Presentation Guidelines

### Deliverables
1. **Source code** — zipped project folder (or GitHub link)
2. **Report** — 5–8 pages covering:
   - System overview and features
   - Database ER diagram
   - Module dependency diagram
   - Screenshots of key pages
   - Individual contribution statement
3. **Live demonstration** — 10–12 minutes

### Demo Structure (10 min)
| Segment | Time |
|---|---|
| Introduction — what problem does it solve? | 1 min |
| Architecture walkthrough (show code) | 2 min |
| Live demo of all features | 5 min |
| Q&A | 2 min |

---
## 8. Self-Study Checklist

Use this checklist to verify your readiness for the final exam.

### Syntax & Fundamentals (Week 1–2)
- [ ] Declare variables of all core types and inspect with `type()`
- [ ] Perform arithmetic, comparison, logical, and assignment operations
- [ ] Format output with f-strings

### Control Flow (Week 3–4)
- [ ] Write if/elif/else with multiple conditions
- [ ] Write for and while loops, use break/continue
- [ ] Implement factorial and Fibonacci from scratch

### Data Structures (Week 5–6)
- [ ] Slice strings and lists with `[start:stop:step]`
- [ ] Write a list comprehension with a filter condition
- [ ] Use set operations: union, intersection, difference

### Modules (Week 7)
- [ ] Import from standard library (math, random, datetime)
- [ ] Create a custom module and import it

### File Handling (Week 8)
- [ ] Read and write a text file using `with open`
- [ ] Read and write a CSV file using `csv.DictReader`/`DictWriter`

### Exceptions (Week 9)
- [ ] Handle ValueError and FileNotFoundError
- [ ] Write a custom exception class
- [ ] Use try/except/else/finally correctly

### Web (Week 10–11)
- [ ] Create Flask routes with URL parameters
- [ ] Render a Jinja2 template with passed data
- [ ] Handle a POST form submission with redirect

### Database (Week 12–13)
- [ ] Create a SQLite table and perform CRUD
- [ ] Write a JOIN query
- [ ] Use parameterised queries (never string concatenation)

---
## 9. Practice Exam Questions

The cells below mirror the style of Section A (Theory) and Section B (Practical) of the final exam.

### Section A — Theory Questions

**Q1.** Explain the difference between `break` and `continue` in a loop. Give one example of each. *(6 pts)*

**Q2.** Compare lists, tuples, and sets: mutability, order, duplicate handling, and typical use case. *(9 pts)*

**Q3.** What is SQL injection? How do parameterised queries prevent it? *(4 pts)*

**Q4.** Describe the role of each clause in `try / except / else / finally`. *(8 pts)*

**Q5.** Explain the Flask request/response cycle for a POST form submission. *(8 pts)*

**Q6.** What is the difference between `conn.commit()` and `conn.rollback()` in SQLite? *(5 pts)*

In [ ]:
# Section B — Q1 (10 pts)
# Write a function prime_factors(n) that returns a list of all prime factors of n.
# Example: prime_factors(360) → [2, 2, 2, 3, 3, 5]

def prime_factors(n):
    # TODO
    pass

print(prime_factors(360))   # [2, 2, 2, 3, 3, 5]
print(prime_factors(97))    # [97]  (prime number)

In [ ]:
# Section B — Q2 (15 pts)
# Given a CSV file 'results.csv' with columns: name, math, english, science
# Read the file, compute each student's average, assign a letter grade,
# and write a new file 'report.csv' with columns: name, average, grade.
# Handle FileNotFoundError and invalid numeric data gracefully.

import csv

# Create sample input
sample = [
    ["name",    "math", "english", "science"],
    ["Alice",   "92",   "88",      "95"],
    ["Bob",     "75",   "N/A",     "70"],   # bad data
    ["Charlie", "88",   "90",      "85"],
]
with open("results.csv", "w", newline="") as f:
    csv.writer(f).writerows(sample)

# TODO: implement the solution


In [ ]:
# Section B — Q3 (20 pts)
# Using SQLite, create a 'library' database with tables:
#   books(id, title, author, year, available)
#   members(id, name, email)
#   loans(id, book_id, member_id, loan_date, return_date)
#
# Implement functions:
#   borrow_book(book_id, member_id)  — marks book unavailable, creates loan
#   return_book(loan_id)             — marks book available, sets return_date
#   overdue_loans()                  — returns loans with no return_date after 14 days

import sqlite3

# TODO


---
## Course Summary — MCS603 Programming II

| Week | Topic | Key Skills |
|---|---|---|
| 1–2 | Fundamentals | Variables, data types, operators, I/O |
| 3–4 | Control Flow | if/elif/else, for/while, break/continue |
| 5–6 | Data Structures | str, list, tuple, set, comprehensions |
| 7 | Modules | import, standard library, custom modules |
| 8 | File Handling | text files, CSV, os.path |
| 9 | Exceptions | try/except/finally, raise, custom exceptions |
| 10–11 | Web Dev | Flask routes, Jinja2 templates, forms |
| 12–13 | Databases | SQLite CRUD, JOIN, parameterised queries |
| 14 | Capstone | Integration: Flask + SQLite + modules + files |

---
> **Congratulations** on completing MCS603 Programming II!  
> Continue your Python journey with MCS604 — Data Structures & Algorithms.